# 1. Introdução ao Processamento de Linguagem Natural: Sumarização de Textos

Seja muito bem-vindo! Este notebook é um guia prático projetado para ensinar como utilizar a biblioteca **Hugging Face Transformers** para realizar a tarefa de **geração de texto e sumarização**. Aqui, você aprenderá desde a instalação das ferramentas necessárias até como contornar limitações de modelos de linguagem para obter resumos coerentes e completos.

### Objetivos de Aprendizagem
* Entender o papel da biblioteca `transformers` e do conceito de `pipeline` no ecossistema de Inteligência Artificial.
* Compreender a diferença prática entre modelos de geração livre de texto e modelos específicos de sumarização.
* Aprender a controlar o comportamento de modelos de linguagem ajustando parâmetros como `max_new_tokens` e `min_length`.
* Desenvolver habilidades de manipulação e fatiamento de strings em Python para limpar saídas geradas por IA.

### Pré-requisitos e Bibliotecas Usadas
* **`transformers`**: Desenvolvida pela Hugging Face, esta biblioteca é o padrão da indústria para trabalhar com modelos de linguagem modernos (como GPT, Llama, SmolLM). Nós a utilizamos para carregar e executar o modelo de Inteligência Artificial de forma simples.
* **Python Básico**: Conhecimentos sobre variáveis, fatiamento de strings (`slicing`) e dicionários são fundamentais para manipular as respostas que a IA nos devolve.

## Passo 1: Instalando a Biblioteca Principal

### O que este passo faz
Instala a biblioteca `transformers` da Hugging Face no ambiente do Google Colab.

### Por que ele é necessário
O Google Colab vem com várias bibliotecas pré-instaladas, mas o `transformers` precisa ser adicionado manualmente para que possamos ter acesso aos modelos de Inteligência Artificial de última geração.

### Como interpretar o resultado
Se a instalação for bem-sucedida, você verá mensagens como `Requirement already satisfied` ou um aviso de que os pacotes foram instalados com sucesso. Isso significa que as ferramentas estão prontas para serem importadas para o nosso código.

In [1]:
!pip install transformers

## Passo 2: Importando as Ferramentas

### O que este passo faz
Importa a biblioteca `transformers` e, especificamente, a função `pipeline`.

### Por que ele é necessário
O **`pipeline`** (em português, *linha de produção* ou *canalização*) é uma ferramenta de altíssimo nível do Hugging Face. Ele abstrai toda a complexidade de processar o texto, carregar o modelo e converter as palavras em números (tokenização), permitindo que você use IA com apenas uma linha de código.

### Como interpretar o resultado
A célula deve rodar rapidamente e sem gerar erros visíveis, indicando que a função agora está disponível na memória do notebook.

In [2]:
import transformers
from transformers import pipeline

## Passo 3: Inicializando o Gerador de Texto

### O que este passo faz
Inicializa um pipeline configurado para a tarefa de **`text-generation`** (geração de texto).

### Por que ele é necessário
Modelos de linguagem modernos usam a geração de texto como base para quase tudo.

💡 **Para pensar**:
O código inicializa o `pipeline("text-generation")` sem especificar um modelo concreto por parâmetro. Quando fazemos isso, o Hugging Face baixa automaticamente um modelo padrão (neste caso, o `SmolLM3-3B`). Em ambientes de produção, é sempre uma boa prática explicitar qual modelo você deseja usar para evitar surpresas ou mudanças inesperadas se o padrão da biblioteca mudar.

### Como interpretar o resultado
Você verá um aviso do sistema informando que nenhum modelo foi fornecido (`No model was supplied`) e que ele escolheu o `HuggingFaceTB/SmolLM3-3B` por padrão. Isso é esperado e indica que o download do modelo foi concluído.

In [ ]:
# Usando o pipeline de text-generation recomendado para a versão 5 do transformers
gerador = pipeline("text-generation")

[transformers] No model was supplied, defaulted to HuggingFaceTB/SmolLM3-3B and revision a07cc9a.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/26.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

## Passo 4: Definindo o Texto de Entrada

### O que este passo faz
Cria uma variável chamada `texto` que armazena uma pequena biografia em texto do cientista Carl Sagan.

### Por que ele é necessário
Antes de pedir para a Inteligência Artificial trabalhar, precisamos fornecer a matéria-prima (os dados) sobre a qual ela deve operar.

### Como interpretar o resultado
O texto é apenas armazenado na memória. Não há saída visual ao executar esta célula.

In [14]:
texto = """Carl Edward Sagan foi um astrônomo, astrofísico, astrobiólogo, cientista planetário, escritor e divulgador científico norte-americano. Embora tenha estudado física e atuado fortemente na astrobiologia, ele não era formado como biólogo — sua área de pesquisa central combinava astronomia, física planetária e a busca por vida fora da Terra."""

## Passo 5: Primeira Tentativa de Geração

### O que este passo faz
Tenta gerar um texto a partir da biografia de Carl Sagan, definindo limites para o tamanho da resposta.

### Por que ele é necessário
Esta etapa serve para demonstrar como o modelo de geração se comporta de forma padrão.

💡 **Para pensar**:
Ao rodar esse trecho, você notará que o modelo não gerou um resumo, mas sim completou o texto original com a frase de continuação. Isso acontece porque o pipeline de geração livre de texto tenta apenas *continuar* o texto fornecido. Para obter um resumo real, precisamos guiar a IA usando uma técnica chamada engenharia de prompt, como veremos no próximo passo.

### Como interpretar o resultado
A saída retornada é uma lista contendo um dicionário com a chave `'generated_text'`. O valor associado é o texto original seguido da continuação do modelo.

In [ ]:
resumo = gerador(texto, max_length=100, min_length=50)
print(resumo)

## Passo 6: Forçando a Sumarização via Instruções (Prompt)

### O que este passo faz
Cria uma instrução explícita combinando o comando de resumo com o texto original, gera o resultado limitando-o a até 200 novos tokens e limpa a resposta utilizando manipulação de strings em Python.

### Por que ele é necessário
Como o modelo de geração sempre retorna tanto a pergunta quanto a resposta combinadas, precisamos de lógica em Python para extrair apenas o resumo:
* **`texto_gerado = resultado[0]['generated_text']`**: Extrai a string bruta de dentro da estrutura de lista/dicionário.
* **`len(prompt)`**: Descobre o tamanho do comando inicial.
* **`[len(prompt):]`**: Usa fatiamento (`slicing`) para cortar o comando original do início da string, preservando apenas o que a IA escreveu a partir dali.
* **`.strip()`**: Remove quebras de linha ou espaços indesejados nas pontas.

💡 **Para pensar**:
Se você reparar na última frase do resumo gerado nesta célula, ela pode terminar de forma abrupta. Isso acontece porque o parâmetro `max_new_tokens=200` atinge seu limite estrito antes que o modelo consiga colocar o ponto final. Para resolver isso de forma elegante, podemos aumentar ainda mais o limite ou instruir o modelo a concluir o pensamento antes do limite.

### Como interpretar o resultado
Você verá um resumo textual conciso sobre a vida de Carl Sagan em português, demonstrando o poder de instrução de modelos modernos de linguagem.

In [ ]:
# Criamos uma instrução clara para o modelo saber que deve resumir
prompt = f"Resuma o seguinte texto em português:\n\n{texto}\n\nResumo:"

# Geramos a resposta com base na instrução
resultado = gerador(prompt, max_new_tokens=200, min_length=20)

# Exibimos apenas a parte gerada após o nosso prompt
texto_gerado = resultado[0]['generated_text']
resumo_final = texto_gerado[len(prompt):].strip()
print(resumo_final)

## 5. Erros Comuns e Dicas

* **`KeyError: 'Unknown task...'`**: Ocorre quando você digita o nome de uma tarefa que não existe na biblioteca (como `sumarization` com um único 'm'). Sempre use termos corretos em inglês, como `summarization` (que foi descontinuada na v5 em favor de modelos conversacionais) ou `text-generation`.
* **Textos Cortados ao Meio**: Se o seu texto final terminar no meio de uma frase, é um sinal clássico de que o valor de `max_new_tokens` é muito pequeno para a complexidade da resposta exigida. Aumente o valor gradativamente.
* **O Modelo Repete a Pergunta**: Lembre-se que modelos de `text-generation` devolvem a entrada (prompt) junta com a saída. O fatiamento `[len(prompt):]` é essencial para isolar o resultado.

---

## 6. Resumo do que foi aprendido
* **Pipelines de IA**: Aprendemos que a função `pipeline` simplifica imensamente o uso de modelos complexos de Inteligência Artificial.
* **Geração vs. Sumarização**: Compreendemos que modelos gerativos precisam de instruções diretas (*prompts*) claras para desempenhar tarefas de formatação específicas, como resumir.
* **Fatiamento de Strings**: Praticamos como manipular as respostas em Python para extrair apenas a informação relevante gerada pela IA.

---

## 7. Exercícios Propostos

### Exercício 1 (Fácil)
Altere o texto contido na variável `texto` para uma curta notícia recente de sua escolha e execute novamente a célula de geração do resumo. Observe se o modelo consegue identificar os pontos principais da sua notícia.

### Exercício 2 (Médio)
Altere o prompt do Passo 6 para que o modelo faça o resumo em formato de **tópicos** (bullet points) em vez de um parágrafo corrido. Ajuste o fatiamento de string se necessário para exibir o resultado limpo.

### Exercício 3 (Desafiador)
Crie uma função em Python chamada `gerar_resumo_curto(texto_entrada)` que receba qualquer texto como parâmetro, monte o prompt internamente, execute o gerador limitando a resposta a no máximo 50 tokens e retorne apenas o resumo fatiado e limpo.

---

## 8. Para ir além
* **Engenharia de Prompt**: Pesquise sobre técnicas de *Few-Shot Prompting* para entender como dar exemplos ao modelo antes de pedir a resposta final.
* **Tokenização**: Pesquise como as palavras são convertidas em pedaços menores chamados *tokens* e como isso afeta a cobrança e os limites de memória dos modelos de IA.
* **Modelos de Chat**: Leia sobre o formato de mensagens estruturadas (com papéis de `system`, `user` e `assistant`) para interagir com modelos de linguagem de forma mais organizada do que textos puros concatenados.